Foram realizados testes com diversas APIs gratuitas para a extração de dados das ações, as mais simples de utilizar não disponibilizam os dados no nível que que precisava, por isso utilizei o MetaTrader5, mas para isso foi necessário instalar ele na minha máquina e salvar os dados.

In [ ]:
!pip install MetaTrader5
!pip install pandas
!pip install matplotlib
!pip install nasdaq-data-link
!pip install pyarrow
!pip install openpyxl

In [ ]:
from datetime import datetime
import matplotlib.pyplot as plt
import pandas as pd
import MetaTrader5 as mt5
import requests
from datetime import datetime, timedelta
from dateutil.relativedelta import relativedelta 
import numpy as np

In [3]:
#Iniciar conexão com o MetaTrader5
def start_mt5(username, password, server, path):
    # Ensure that all variables are the correct type
    uname = int(username) # Username must be an int
    pword = str(password) # Password must be a string
    trading_server = str(server) # Server must be a string
    filepath = str(path) # Filepath must be a string

    # Attempt to start MT5
    if mt5.initialize(login=uname, password=pword, server=trading_server, path=path):
        # Login to MT5
        if mt5.login(login=uname, password=pword, server=trading_server):
            return True
        else:
            print("Login Fail")
            quit()
            return PermissionError
    else:
        print("MT5 Initialization Failed")
        quit()
        return ConnectionAbortedError

In [4]:
#Encerrar conexão com o MetaTrader5
def end_mt5():
    mt5.shutdown()

In [ ]:
#Criar arquivos parquet com os dados históricos desde 2021 do MetaTrader5
start_mt5(4003595,"*****","mt5.xpi.com.br:443","")

#Extrair dados dos índices
rates = pd.DataFrame()
for a in range(2023,2026):
    rates = pd.concat([rates,pd.DataFrame(mt5.copy_rates_range("PETR4",mt5.TIMEFRAME_M15,datetime(a,1,1,0,0,0,0),datetime(a,12,31,23,59,59,999999)))],ignore_index=True)
rates.to_parquet("petr4.parquet")

#Extrair dados dos ticks
ticks = pd.DataFrame()
for a in range(2023,2026):
    for m in range(1,13): 
        temp = pd.DataFrame(mt5.copy_ticks_range("PETR4", datetime(a,m,1,0,0,0,0),(datetime(a,m,1,0,0,0,0)+relativedelta(months=1)-timedelta(milliseconds=0.001)),mt5.COPY_TICKS_ALL))
        temp[temp.select_dtypes(np.float64).columns] = temp.select_dtypes(np.float64).astype(np.float32)
        ticks = pd.concat([ticks,temp],ignore_index=True)
ticks.to_parquet("petr4_ticks.parquet")

end_mt5()

In [ ]:
if mt5.market_book_add('PETR4'):
    mt5.market_book_get('PETR4')
mt5.market_book_release()


In [9]:
#Extrair dados do brent do Alpha Vantage
url = "https://www.alphavantage.co/query?function=BRENT&interval=daily&apikey=FYMI9F6YGAAAVUZ6"
brent = requests.get(url)
brent = pd.DataFrame(brent.json()["data"])
brent.to_parquet("brent.parquet")

#Extrair dados diários do indice do Alpha Vantage
url = "https://www.alphavantage.co/query?function=TIME_SERIES_DAILY&symbol=PETR4.SAO&apikey=FYMI9F6YGAAAVUZ6"
rates = requests.get(url)
rates = pd.DataFrame(rates.json()["Time Series (Daily)"]).transpose()
rates.to_parquet("petr4_daily.parquet")